# Problem Statement

The UCI News Aggregator dataset is a set of over 420,000 news articles that were compiled in 2014.  

Our goal for this analysis is to determine underlying trends among the articles to uncover commonalities and other links within the dataset using K-nearest neighbors, a machine learning technique that's usually used in analyzing unlabeled data.  However, in this case, it will be used to aid in further exploring the UCI News Aggregator dataset to uncover trends that we may not notice otherwise.

# Dictionary

- ID: the numeric ID of the article
- TITLE: the headline of the article
- URL: the URL of the article
- PUBLISHER: the publisher of the article
- CATEGORY: the category of the news item; one of:
    - e: entertainment
    - b: business
    - t: science and technology
    - m: health
- STORY: alphanumeric ID of the news story that the article discusses
- HOSTNAME: hostname where the article was posted
- TIMESTAMP: approximate timestamp of the article's publication, given in Unix time (seconds since midnight on Jan 1, 1970)

# Exploration

In the data exploration phase, we will cleanse the data to ensure suitable usage for modelling.

## Library Imports

We will import the NumPy, Pandas, Scikit-Learn Matplotlib, and Seaborn libraries to analyze, model and visualize our data.

Meanwhile, pickle and re will allow us to save our variables and perform internal operations within the system in which we perform this analysis.

In [ ]:
# !pip install seaborn
# !pip install --upgrade pip

In [ ]:
# Import the packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re

from sklearn.feature_extraction.text import CountVectorizer

import pickle

import warnings
warnings.filterwarnings("ignore")

# Import the SentenceBERT model
from sentence_transformers import SentenceTransformer

# Classification model
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

# Post analysis
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.manifold import TSNE

# Import the 3D visualization libraries
from mpl_toolkits.mplot3d import Axes3D
import plotly.express as px
import plotly.io as pio

from sklearn.utils import resample

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

data_path = '/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/data/'
results_path = '/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/results/'

Mounted at /content/drive


In [ ]:
# Import the datasets
dataset = pd.read_csv('/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/data/convabuse_sa_punc_imbalanced.csv')

# News Title Classifier

Now that we've explored the data in-depth, we can move forward to analyze the data and model a classifier to predict the classification of a news article based on its title.

This is, once again, a preview of the dataset after preprocessing.

In [ ]:
dataset.head()

,annotator_id,prev_agent,prev_user,agent,user,bot,generalised,individual,system,explicit,implicit,abuse_level,target_groups,text,prev_agent_length,prev_user_length,agent_length,user_length,text_length,special_characters
0,7,Does that question interest you?,because my finger is happy,Why do you say your finger is happy?,do you rember japan,0,0,0,0,0,0,0,0,Does that question interest you? because my fi...,5,5,8,4,22,False
1,7,Does it please you to believe I am a nobe jocky?,bye bye,Please go on.,do you like it hard,0,0,0,0,0,0,1,0,Does it please you to believe I am a nobe jock...,11,2,3,5,21,False
2,7,Have you tried?,dick,Please go on.,oh what a pickle dear monastery,0,0,0,0,0,0,0,0,Have you tried? dick Please go on. oh what a p...,3,1,3,6,13,False
3,7,Will you be travelling in Economy Class?,yes,What city will you be flying from?,Sydney,1,0,0,0,0,0,0,0,Will you be travelling in Economy Class? yes W...,7,1,7,1,16,True
4,7,Please go on.,Hello.,How do you do. Please state your problem.,how old are you?,0,0,0,0,0,0,0,0,Please go on. Hello. How do you do. Please sta...,3,1,8,4,16,False


## Univariate Modeling

Only use the title as the independent variable and the category as the dependent variable.

In [ ]:
dataset.columns

Index(['annotator_id', 'prev_agent', 'prev_user', 'agent', 'user', 'bot',
       'generalised', 'individual', 'system', 'explicit', 'implicit',
       'abuse_level', 'target_groups', 'text', 'prev_agent_length',
       'prev_user_length', 'agent_length', 'user_length', 'text_length',
       'special_characters'],
      dtype='object')

In [ ]:
imbalanced = dataset[['annotator_id', 'bot', 'generalised', 'individual', 'system', 'explicit', 'implicit', 'abuse_level', 'target_groups', 'prev_agent', 'prev_user', 'agent', 'user']]
imbalanced.head()

,annotator_id,bot,generalised,individual,system,explicit,implicit,abuse_level,target_groups,prev_agent,prev_user,agent,user
0,7,0,0,0,0,0,0,0,0,Does that question interest you?,because my finger is happy,Why do you say your finger is happy?,do you rember japan
1,7,0,0,0,0,0,0,1,0,Does it please you to believe I am a nobe jocky?,bye bye,Please go on.,do you like it hard
2,7,0,0,0,0,0,0,0,0,Have you tried?,dick,Please go on.,oh what a pickle dear monastery
3,7,1,0,0,0,0,0,0,0,Will you be travelling in Economy Class?,yes,What city will you be flying from?,Sydney
4,7,0,0,0,0,0,0,0,0,Please go on.,Hello.,How do you do. Please state your problem.,how old are you?


In [ ]:
# Merge the prev_agent_text, prev_user_text, agent_text, and user_text columns
imbalanced["text"] = imbalanced["prev_agent"].astype(str) + " " + imbalanced["prev_user"].astype(str) + " " + imbalanced["agent"].astype(str) + " " + imbalanced["user"].astype(str)

# Preview the DataFrame
imbalanced.head()

,annotator_id,bot,generalised,individual,system,explicit,implicit,abuse_level,target_groups,prev_agent,prev_user,agent,user,text
0,7,0,0,0,0,0,0,0,0,Does that question interest you?,because my finger is happy,Why do you say your finger is happy?,do you rember japan,Does that question interest you? because my fi...
1,7,0,0,0,0,0,0,1,0,Does it please you to believe I am a nobe jocky?,bye bye,Please go on.,do you like it hard,Does it please you to believe I am a nobe jock...
2,7,0,0,0,0,0,0,0,0,Have you tried?,dick,Please go on.,oh what a pickle dear monastery,Have you tried? dick Please go on. oh what a p...
3,7,1,0,0,0,0,0,0,0,Will you be travelling in Economy Class?,yes,What city will you be flying from?,Sydney,Will you be travelling in Economy Class? yes W...
4,7,0,0,0,0,0,0,0,0,Please go on.,Hello.,How do you do. Please state your problem.,how old are you?,Please go on. Hello. How do you do. Please sta...


In [ ]:
# Drop the previous columns
imbalanced = imbalanced.drop(columns=['prev_agent', 'prev_user', 'agent', 'user'])
imbalanced.head()

,annotator_id,bot,generalised,individual,system,explicit,implicit,abuse_level,target_groups,text
0,7,0,0,0,0,0,0,0,0,Does that question interest you? because my fi...
1,7,0,0,0,0,0,0,1,0,Does it please you to believe I am a nobe jock...
2,7,0,0,0,0,0,0,0,0,Have you tried? dick Please go on. oh what a p...
3,7,1,0,0,0,0,0,0,0,Will you be travelling in Economy Class? yes W...
4,7,0,0,0,0,0,0,0,0,Please go on. Hello. How do you do. Please sta...


Apply the train/test split in preparation for modeling.

In [ ]:
# X and y
X = imbalanced.drop(columns=['abuse_level'])
y = imbalanced[['abuse_level']]

In [ ]:
X.head()

,annotator_id,bot,generalised,individual,system,explicit,implicit,target_groups,text
0,7,0,0,0,0,0,0,0,Does that question interest you? because my fi...
1,7,0,0,0,0,0,0,0,Does it please you to believe I am a nobe jock...
2,7,0,0,0,0,0,0,0,Have you tried? dick Please go on. oh what a p...
3,7,1,0,0,0,0,0,0,Will you be travelling in Economy Class? yes W...
4,7,0,0,0,0,0,0,0,Please go on. Hello. How do you do. Please sta...


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Train/test split
X_train_titles, X_test_titles, y_train_titles, y_test_titles = X_train['text'], X_test['text'], y_train['abuse_level'], y_test['abuse_level']

# Check the distribution of the dependent variable in train vs test
train_dist = X_train['text'].value_counts(normalize=True)
test_dist = X_train['text'].value_counts(normalize=True)
print("Train Distribution:")
print(train_dist)
print("\nTest Distribution:")
print(test_dist)

Train Distribution:
text
_ _ _ hi                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     0.002286
_ _ _ Hi                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [ ]:
pd.Series(y_train_titles).value_counts()

,count
abuse_level,
0,8050
3,694
2,599
1,526
4,192


## Embed Text

The sentence transformer (in this case, SentenceBERT) allows us to see the embeddings, which are vector representations of the text.

In [ ]:
# Install the sentence transformers
# !pip install -U sentence-transformers

Here's a function that will convert our text into embeddings to lower dimensionality.

In [ ]:
# Develop a get_embeddings function to get the embeddings using the SentenceBERT model to embed the text
    # The embeddings are the vector representations of the text
def get_embeddings(text):
    """
    Get the embeddings for the text using the SentenceBERT model.
    """
    # Load the SentenceBERT model
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Get the embeddings
    embeddings = model.encode(text)

    return embeddings

In [ ]:
results_path = '/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/results/'

Generate the embeddings for the title text as the dependent variables.

In [ ]:
# Build merged text for each split
X_train_titles = X_train_titles.copy()
X_test_titles  = X_test_titles.copy()

# The original code was trying to create a new column 'merged' on a Series,
# which is not a valid operation for a Series and also unnecessary
# as 'convabuse_X_train_titles' already contains the combined text.
# The `agg(" ".join, axis=1)` is specifically for DataFrames to operate across columns,
# but 'convabuse_X_train_titles' is a Series (1-dimensional), hence 'axis=1' is invalid.

# Embed directly using the content of the Series
X_train_emb = get_embeddings(X_train_titles.tolist())
X_test_emb  = get_embeddings(X_test_titles.tolist())

# Align labels to the same indices
y_train_emb = y_train_titles.reindex(X_train_titles.index)
y_test_emb  = y_test_titles.reindex(X_test_titles.index)

# Convert to numpy for sklearn
y_train_emb = y_train.to_numpy()
y_test_emb  = y_test.to_numpy()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Apply UMAP
import umap
reducer = umap.UMAP()
X_train_umap = reducer.fit_transform(X_train_emb)
X_test_umap  = reducer.fit_transform(X_test_emb)
y_train_umap = reducer.fit_transform(y_train_emb)
y_test_umap = reducer.fit_transform(y_test_emb)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((10061, 9), (2516, 9), (10061, 1), (2516, 1))

In [ ]:
X_train_umap.shape, X_test_umap.shape, y_train_umap.shape, y_test_umap.shape

((10061, 2), (2516, 2), (10061, 2), (2516, 2))

In [ ]:
# For each entry in X_train, X_test, y_train, and y_test, remove the final element
X_train_edited = np.delete(X_train, -1, axis=1)
X_test_edited = np.delete(X_test, -1, axis=1)

In [ ]:
# Append the UMAP dimensions to their respective sets
X_train_combined = np.concatenate((X_train_edited, X_train_umap), axis=1)
X_test_combined  = np.concatenate((X_test_edited, X_test_umap), axis=1)
y_train_combined = np.concatenate((y_train_emb, y_train_umap), axis=1)
y_test_combined = np.concatenate((y_test_emb, y_test_umap), axis=1)

In [ ]:
X_train_combined[0]

array([1, 1, 0, 0, 1, 1, 0, 0, -4.554397106170654, -7.792840480804443],
      dtype=object)

In [ ]:
# Organize into a dataframe with columns annotator_id, bot, generalised, individual, system, explicit, implicit, target_groups
X_trained_combined_df = pd.DataFrame(X_train_combined, columns=['annotator_id', 'bot', 'generalised', 'individual', 'system', 'explicit', 'implicit', 'target_groups', 'umap_1', 'umap_2'])
X_test_combined_df = pd.DataFrame(X_test_combined, columns=['annotator_id', 'bot', 'generalised', 'individual', 'system', 'explicit', 'implicit', 'target_groups', 'umap_1', 'umap_2'])

In [ ]:
X_trained_combined_df.head()

,annotator_id,bot,generalised,individual,system,explicit,implicit,target_groups,umap_1,umap_2
0,1,1,0,0,1,1,0,0,-4.554397,-7.79284
1,3,1,0,0,0,0,0,0,9.674322,6.133567
2,4,0,0,0,0,0,0,0,1.651397,1.773538
3,5,0,0,0,0,0,0,0,4.557061,4.054142
4,5,0,0,0,0,0,0,0,0.529315,-2.037056


In [ ]:
# Combine into a single dataframe
combined_X = pd.concat([X_trained_combined_df, X_test_combined_df], axis=0)
combined_y = pd.concat([y_train, y_test], axis=0)

In [ ]:
combined_X.head()

,annotator_id,bot,generalised,individual,system,explicit,implicit,target_groups,umap_1,umap_2
0,1,1,0,0,1,1,0,0,-4.554397,-7.79284
1,3,1,0,0,0,0,0,0,9.674322,6.133567
2,4,0,0,0,0,0,0,0,1.651397,1.773538
3,5,0,0,0,0,0,0,0,4.557061,4.054142
4,5,0,0,0,0,0,0,0,0.529315,-2.037056


In [ ]:
combined_X.shape, combined_y.shape

((12577, 10), (12577, 1))

In [ ]:
# Pickle the embeddings
pickle_path = '/content/drive/My Drive/Online MSDS/MOD C2/Political Polarization/pickle/'

with open(f'{pickle_path}convabuse_combined_imbalanced_X.pkl', 'wb') as f:
    pickle.dump(combined_X, f)
with open(f'{pickle_path}convabuse_combined_imbalanced_y.pkl', 'wb') as f:
    pickle.dump(combined_y, f)

In [ ]:
print(type(X_train_titles))

<class 'pandas.core.series.Series'>


In [ ]:
# Output the titles as a csv file
X_train_titles.to_csv(f'{results_path}convabuse_X_train_titles_combined_new_imbalanced.csv', index=False)
X_test_titles.to_csv(f'{results_path}convabuse_X_test_titles_combined_new_imbalanced.csv', index=False)
y_train_titles.to_csv(f'{results_path}convabuse_y_train_titles_combined_new_imbalanced.csv', index=False)
y_test_titles.to_csv(f'{results_path}convabuse_y_test_titles_combined_new_imbalanced.csv', index=False)